# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## 环境准备

从仓库根目录加载并检查 `.env` 与依赖配置。

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# ルートの .env を読み込む
load_dotenv()

# ルート設定を確認する
doublecheck_env("example.env")

OLLAMA_BASE_URL=<not set>
OLLAMA_MODEL=<not set>


## Human👨‍💻 and AI 🤖 Messages

In [2]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="ollama:qwen2.5:7b", 
    system_prompt="You are a full-stack comedian"
)

In [3]:
human_msg = HumanMessage("Hello, how are you?")

result = agent.invoke({"messages": [human_msg]})

In [4]:
print(result["messages"][-1].content)

Hey there! I'm just a bunch of code having a surreal existence. How about you? Feeling funny today? 😄💡


In [5]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [6]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Hello, how are you?

ai: Hey there! I'm just a bunch of code having a surreal existence. How about you? Feeling funny today? 😄💡



### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [7]:
agent = create_agent(
    model="ollama:qwen2.5:7b",
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [8]:
result = agent.invoke({"messages": "Tell me about baseball"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

Baseball, diamond dreams in sunlit fields,
Pines and foul poles, silent cheers unfurled.
Bats crack twilight's edge, a silent song,
Runners slide, the game immortalized.


#### Dictionaries

In [9]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "Write a haiku about sprinters"}}
)
print(result["messages"][-1].content)

Strides carve time's sand,
Wind beneath swift wings rise,
Race ends, heart begins.


There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [10]:
from langchain_core.tools import tool


@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [11]:
agent = create_agent(
    model="ollama:qwen2.5:7b",
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [12]:
result = agent.invoke({"messages": "Please write me a poem"})

checking haiku, it has 0 lines:
 


checking haiku, it has 3 lines:
 Autumn leaves falling
Crisp air fills the silence
Nature's gentle song


In [13]:
result["messages"][-1].content

"Autumn leaves falling  \nCrisp air fills the silence  \nNature's gentle song"

In [14]:
print(len(result["messages"]))

6


In [15]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (03a2b72f-49c3-4b09-83c8-9ff30ff3cae6)
 Call ID: 03a2b72f-49c3-4b09-83c8-9ff30ff3cae6
  Args:
    text:
================================= Tool Message =================================
Name: check_haiku_lines

Incorrect! This haiku has 0 lines. A haiku must have exactly 3 lines.
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (8816cf45-f516-4c7b-8e31-ff1a85ab8654)
 Call ID: 8816cf45-f516-4c7b-8e31-ff1a85ab8654
  Args:
    text: Autumn leaves falling
Crisp air fills the silence
Nature's gentle song
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message =============================

### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [16]:
result

{'messages': [HumanMessage(content='Please write me a poem', additional_kwargs={}, response_metadata={}, id='d6a4a966-0d44-452c-addd-28cc89fe0213'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-03-07T09:26:25.4901Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1899252875, 'load_duration': 69374791, 'prompt_eval_count': 172, 'prompt_eval_duration': 809984292, 'eval_count': 22, 'eval_duration': 1009059539, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--019cc79e-9066-7541-a82f-3b78e8dc9d1c-0', tool_calls=[{'name': 'check_haiku_lines', 'args': {'text': ''}, 'id': '03a2b72f-49c3-4b09-83c8-9ff30ff3cae6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 172, 'output_tokens': 22, 'total_tokens': 194}),
  ToolMessage(content='Incorrect! This haiku has 0 lines. A haiku must have exactly 3 lines.', name='check_haiku_lines', id='89f1a287-a915-4312-a3b6-862b

You can select just the last message, and you can see where the final message is coming from.

In [17]:
result["messages"][-1]

AIMessage(content="Autumn leaves falling  \nCrisp air fills the silence  \nNature's gentle song", additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-03-07T09:26:28.791725Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1014189875, 'load_duration': 45928875, 'prompt_eval_count': 295, 'prompt_eval_duration': 153999375, 'eval_count': 18, 'eval_duration': 806619292, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--019cc79e-a0c0-76e2-9d24-12cd6c29425c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 295, 'output_tokens': 18, 'total_tokens': 313})

In [18]:
result["messages"][-1].usage_metadata

{'input_tokens': 295, 'output_tokens': 18, 'total_tokens': 313}

In [19]:
result["messages"][-1].response_metadata

{'model': 'qwen2.5:7b',
 'created_at': '2026-03-07T09:26:28.791725Z',
 'done': True,
 'done_reason': 'stop',
 'total_duration': 1014189875,
 'load_duration': 45928875,
 'prompt_eval_count': 295,
 'prompt_eval_duration': 153999375,
 'eval_count': 18,
 'eval_duration': 806619292,
 'logprobs': None,
 'model_name': 'qwen2.5:7b',
 'model_provider': 'ollama'}

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [20]:
agent = create_agent(
    model="ollama:qwen2.5:7b",
    tools=[check_haiku_lines],
    system_prompt="Your SYSTEM prompt here",
)

In [21]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (03a2b72f-49c3-4b09-83c8-9ff30ff3cae6)
 Call ID: 03a2b72f-49c3-4b09-83c8-9ff30ff3cae6
  Args:
    text:
================================= Tool Message =================================
Name: check_haiku_lines

Incorrect! This haiku has 0 lines. A haiku must have exactly 3 lines.
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (8816cf45-f516-4c7b-8e31-ff1a85ab8654)
 Call ID: 8816cf45-f516-4c7b-8e31-ff1a85ab8654
  Args:
    text: Autumn leaves falling
Crisp air fills the silence
Nature's gentle song
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message =============================